In [2]:
#Importing necessary libraries
from google.oauth2 import service_account
import boto3
import base64
import os
import json
import requests
import urllib.request
import pandas as pd
#import geopandas as gpd
#from shapely import wkt
import re

In [3]:
# Imagine secrets
SECRET_NAME = "veg-dpd-iae-nonprod-dp-sagemaker-secrets"
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")

def get_secrets() -> dict:
    sm = boto3.client("secretsmanager", region_name=AWS_REGION)
    resp = sm.get_secret_value(SecretId=SECRET_NAME)
    return json.loads(resp["SecretString"]) if "SecretString" in resp \
           else json.loads(base64.b64decode(resp["SecretBinary"]))

secrets = get_secrets()
CLIENT_ID     = secrets.get("SCALECAM_IMAGINE_AZURE_APP_ID")
CLIENT_SECRET = secrets.get("SCALECAM_IMAGINE_AZURE_APP_SECRET")

In [4]:
TENANT_ID = "fcb2b37b-5da0-466b-9b83-0014b67a7c78"
SCOPE_ID_PROD = "aa6faa49-d028-4185-b358-75c556add00c"
SCOPE_ID_NP = "73881c8a-db54-4447-a0cf-a5d1c4a1df8d"

OAUTH_TOKEN_URL = f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token"

In [5]:
# Define the get_token() function. Function to generate token without running token server.
def get_token(type = 'PROD'):

    if type == 'PROD':
      payload=f'grant_type=client_credentials&client_id={CLIENT_ID}&client_secret={CLIENT_SECRET}&scope={SCOPE_ID_PROD}%2F.default'
    elif type == 'NP':
      payload=f'grant_type=client_credentials&client_id={CLIENT_ID}&client_secret={CLIENT_SECRET}&scope={SCOPE_ID_NP}%2F.default'
    else:
      raise Exception("Wrong environment type provided. Accepted options: PROD and NP")
    
    headers = {
      'Content-Type': 'application/x-www-form-urlencoded'
    }

    response = requests.request("POST", OAUTH_TOKEN_URL, headers=headers, data=payload)
    
    return response.json()['access_token']

In [6]:
#API's base URLs

#Production environment
IMAGINE_API_BASE_URL = 'https://v4.imagine-api.location360.ag'

#for Non-production environment use variable as below
#IMAGINE_API_BASE_URL = 'https://v4.imagine-api.np.location360.ag'

#Image-as-Array API
IMAGE_AS_ARRAY_API_BASE_URL = 'https://imagery-array-api.location360.ag'

In [7]:
#Generating header with token for all requests (for NP env, type `get_token(NP)`)

headers = {'Authorization': f'Bearer {get_token()}','Content-type': 'application/json'}

In [8]:
#GET list of 10 existing albums to test token and entitlements, optional

url = IMAGINE_API_BASE_URL + '/albums'
response = requests.get(url, headers=headers).json().get('data')
coll_list = [i['name'] for i in requests.get(url, headers=headers).json().get('data')]
print(json.dumps(coll_list[:10], indent=2))

[
  "NAPR_Test",
  "NAPR_Test_2024",
  "Download album",
  "202310 P4s ",
  "6804",
  "CHA ANA GAL ESL Melon",
  "KADABRA ",
  "SVTD8090",
  "Kunzing Tassel Pics 2024",
  "NAPR MIR162 Study 2024 L02"
]


In [9]:
album_id = "aa457dd9-ca7b-459f-80d3-5273c40284e3" #VEG-DSE-DP-TOMATO_LARGE-CALYX-ADO_311047

url = f"{IMAGINE_API_BASE_URL}/albums/{album_id}"
response = requests.get(url, headers=headers)

In [10]:
items = response.json()

In [11]:
print(f"Returned {len(items)} items")

Returned 6 items


In [12]:
print(items)

{'description': 'Annotation dataset for Tomato Large - Calyx. ADO ticket #311047.', 'name': 'VEG-DSE-DP-TOMATO_LARGE-CALYX-ADO_311047', 'created_at': '2025-05-08T21:02:18+00:00', 'created_by': 'GODLY', 'entitlements': {'ADMIN': {'app_ids': ['5083c84f-3d35-4dc0-b0f1-16c35a468b99'], 'papi_groups': ['VEGETABLES-RD-DATA-SCIENCE-AND-ANALYTICS-DIGITAL-PHENOTYPING']}, 'DELETE': {'app_ids': [], 'papi_groups': []}, 'PUBLISH': {'app_ids': [], 'papi_groups': []}, 'DOWNLOAD': {'app_ids': [], 'papi_groups': []}}, 'id': 'aa457dd9-ca7b-459f-80d3-5273c40284e3'}


In [13]:
response = requests.get(
    f"{IMAGINE_API_BASE_URL}/albums/{album_id}/items",
    headers=headers,
    params={"limit": 500}
)

data = response.json()

print(data.keys())

dict_keys(['data', 'total_count'])


In [14]:
#Print number of items in the album
print(data['total_count'])

231


In [15]:
#Print the keys of the items (item number 0)
print(data['data'][0].keys())

dict_keys(['type', 'stac_version', 'stac_extensions', 'id', 'collection', 'properties', 'links', 'assets', 'geometry'])


In [16]:
#Print the collection_id of the items in the album
print(data['data'][0]['collection'])
collection_id = data['data'][0]['collection']

vegetables_plant_images


In [17]:
#Print the item_id of item 0
print(data['data'][0]['id'])

a0100123-cd33-431b-9d86-c6fdfa9f77b7


In [18]:
#Look for the filename
print(data['data'][0]['properties'].keys())

dict_keys(['datetime', 'license', 'gh:geohash', 'vegplantimages:pmc', 'vegplantimages:img_id', 'vegplantimages:season', 'vegplantimages:weight', 'vegplantimages:user_id', 'vegplantimages:batch_id', 'vegplantimages:plot_bid', 'vegplantimages:uploaded', 'vegplantimages:bad_batch', 'vegplantimages:bayer_org', 'vegplantimages:crop_name', 'vegplantimages:file_name', 'vegplantimages:field_name', 'vegplantimages:identifier', 'vegplantimages:originator', 'vegplantimages:weight_uom', 'vegplantimages:rep_barcode', 'vegplantimages:feature_type', 'vegplantimages:pixel_size_x', 'vegplantimages:pixel_size_y', 'vegplantimages:resolution_x', 'vegplantimages:resolution_y', 'vegplantimages:testset_name', 'vegplantimages:location_name', 'vegplantimages:platform_type', 'vegplantimages:serial_number', 'vegplantimages:timestamp_utc', 'vegplantimages:platform_version', 'vegplantimages:trait_recipe_name', 'vegplantimages:platform_ser_number', 'project:name', 'created', 'updated'])


In [19]:
#Print the filename of item 0
print(data['data'][0]['properties']['vegplantimages:file_name'])

P00000000879565588619405_250205_123235_685713.png


In [20]:
#Construct a list of all item_ids
item_id_list = []
for item in data['data']:
    item_id = item['id']
    item_id_list.append(item_id)
print(item_id_list)

['a0100123-cd33-431b-9d86-c6fdfa9f77b7', 'ce513204-ef61-43f5-985f-992211293c12', '0c5805fc-a48c-4d76-a061-245c901e61a3', 'e35c1b42-c525-49dc-a62a-da5e9ba62142', 'cd73c26d-fbd9-43ca-8c15-0aa8a7f0fab8', '8c0b03d0-dfd0-4672-aa18-c6d65bce9adb', 'caa71b73-6c84-4b04-837c-2db60299c6f2', '1bd4adf4-a3a4-4cac-9001-b2d96dfac632', '33059dc2-23ff-4ad6-8ee1-7fd3d2db8b36', '799ca6ca-e887-4cff-bb16-1a1835b80ca9', '07e301d2-1f4a-4577-9de5-736dc3049190', '0eb1f81d-7e34-4e58-9458-58298a031379', '60db7a0c-40a2-44d3-b06e-2049e6de323a', '788bce6f-8472-40b8-bbe9-f3e2cf691d93', '31a526d8-e055-490b-956e-45fae91376b1', '6d549127-2e40-485b-8b33-43054e56fa62', '641daf49-5d08-4d8a-acf9-8f908ebf8809', '4a894dfa-2e4b-433c-85b3-45a65d2f00be', 'd1155016-83d1-4be2-916e-d3d90676971a', '4c55bd67-c417-4932-9bfd-11b345f74c8e', 'de1b3ef0-dad3-48ef-8d9d-cefdf65d755d', '61bb4ed1-49a3-442f-84c4-0945aa003ada', '1eaa66ef-f91a-4a83-9bf8-e582ec8c81ba', 'c4a4c9ad-7747-4439-a92b-6819ba28ebdc', 'acdf3c1e-11b7-40d1-95bd-303998c554ec',

## Match images 
### Create a new COCO JSON with filenames matching the images folder. After converting from LS to COCO, now we need to match the filenames.

In [21]:
#Create a list of all filenames
filename_list = []
for item in data['data']:
    filename = item['properties']['vegplantimages:file_name']
    filename_list.append(filename)
print(filename_list)

['P00000000879565588619405_250205_123235_685713.png', 'P00000000879565588619405_250205_123150_709578.png', 'P00000000879565588619405_250205_123025_983640.png', 'P00000000879565588619405_250205_113200_981570.png', 'P00000000879565588619405_250205_112342_010602.png', 'P00000000879565588619405_250205_112242_144018.png', 'P00000000879565588619405_250205_111844_642390.png', 'P00000000879565588619405_250205_111100_714650.png', 'P00000000879565588619405_250205_110602_765147.png', 'P00000000879565588619405_250205_122942_203862.png', 'P00000000879565588619405_250205_122910_465629.png', 'P00000000879565588619405_250205_122833_084524.png', 'P00000000879565588619405_250205_122727_503470.png', 'P00000000879565588619405_250205_122607_009388.png', 'P00000000879565588619405_250205_122517_202040.png', 'P00000000879565588619405_250205_122428_229805.png', 'P00000000879565588619405_250205_122305_350197.png', 'P00000000879565588619405_250205_122205_329333.png', 'P00000000879565588619405_250205_121812_62577

In [22]:
#Create a list pairing item_ids and filenames
image_data_dict = {}
for item_id, filename in zip(item_id_list, filename_list):
    image_data_dict[item_id] = filename
print(image_data_dict)

{'a0100123-cd33-431b-9d86-c6fdfa9f77b7': 'P00000000879565588619405_250205_123235_685713.png', 'ce513204-ef61-43f5-985f-992211293c12': 'P00000000879565588619405_250205_123150_709578.png', '0c5805fc-a48c-4d76-a061-245c901e61a3': 'P00000000879565588619405_250205_123025_983640.png', 'e35c1b42-c525-49dc-a62a-da5e9ba62142': 'P00000000879565588619405_250205_113200_981570.png', 'cd73c26d-fbd9-43ca-8c15-0aa8a7f0fab8': 'P00000000879565588619405_250205_112342_010602.png', '8c0b03d0-dfd0-4672-aa18-c6d65bce9adb': 'P00000000879565588619405_250205_112242_144018.png', 'caa71b73-6c84-4b04-837c-2db60299c6f2': 'P00000000879565588619405_250205_111844_642390.png', '1bd4adf4-a3a4-4cac-9001-b2d96dfac632': 'P00000000879565588619405_250205_111100_714650.png', '33059dc2-23ff-4ad6-8ee1-7fd3d2db8b36': 'P00000000879565588619405_250205_110602_765147.png', '799ca6ca-e887-4cff-bb16-1a1835b80ca9': 'P00000000879565588619405_250205_122942_203862.png', '07e301d2-1f4a-4577-9de5-736dc3049190': 'P00000000879565588619405_250

In [23]:
#Load the COCO JSON file as a dictionary
with open('./images/coco_new.json', 'r') as file:
    data_d = json.load(file)
print(data_d.keys())

dict_keys(['images', 'annotations', 'categories'])


In [24]:
#Map the item_ids to the filenames
count = 0
for indx, image in enumerate(data_d['images']):
    for item_id in item_id_list:
        if item_id in image['file_name']:
            print(item_id, image['file_name'])
            data_d['images'][indx]['file_name'] = image_data_dict[item_id]
            print(item_id, data_d['images'][indx]['file_name'])
            count += 1
#print(count)

33059dc2-23ff-4ad6-8ee1-7fd3d2db8b36 imagine-asset?ref=imagine:prod:33059dc2-23ff-4ad6-8ee1-7fd3d2db8b36:png
33059dc2-23ff-4ad6-8ee1-7fd3d2db8b36 P00000000879565588619405_250205_110602_765147.png
0c5805fc-a48c-4d76-a061-245c901e61a3 imagine-asset?ref=imagine:prod:0c5805fc-a48c-4d76-a061-245c901e61a3:png
0c5805fc-a48c-4d76-a061-245c901e61a3 P00000000879565588619405_250205_123025_983640.png
a0100123-cd33-431b-9d86-c6fdfa9f77b7 imagine-asset?ref=imagine:prod:a0100123-cd33-431b-9d86-c6fdfa9f77b7:png
a0100123-cd33-431b-9d86-c6fdfa9f77b7 P00000000879565588619405_250205_123235_685713.png
61bb4ed1-49a3-442f-84c4-0945aa003ada imagine-asset?ref=imagine:prod:61bb4ed1-49a3-442f-84c4-0945aa003ada:png
61bb4ed1-49a3-442f-84c4-0945aa003ada P00000000879565588619405_250205_113330_303989.png
788bce6f-8472-40b8-bbe9-f3e2cf691d93 imagine-asset?ref=imagine:prod:788bce6f-8472-40b8-bbe9-f3e2cf691d93:png
788bce6f-8472-40b8-bbe9-f3e2cf691d93 P00000000879565588619405_250205_122607_009388.png
cd73c26d-fbd9-43ca-8

In [25]:
#Dump the modified dictionary in a new JSON
with open("./images/coco_new_matched.json", "w") as file:
    json.dump(data_d, file)

## Equalize the quantity of annotations to images
## Now we need to equate the number of annotations in the COCO JSON to the number of image files.

In [28]:
#Get a list of all images in the image folder
list_of_images = []
folder = './images'
contents = os.listdir(folder)
for filename in contents:
    #print(filename)
    if 'png' in filename:
        list_of_images.append(filename)
#print(list_of_image_files)
print(len(list_of_images))

188


In [29]:
#Create a list of the filenames in the annotation file
list_of_filenames_in_annotation = [item['file_name'] for item in data_d['images']]
print(len(list_of_filenames_in_annotation))

122


In [30]:
#Get the filenames that dont have an image
n = 0
list_of_missing_images = []
for filename in list_of_filenames_in_annotation:
    if filename not in list_of_images:
        list_of_missing_images.append(filename)
        n += 1
print(n)
print(list_of_missing_images)

for key, value in image_data_dict.items():
    pass

0
[]


In [31]:
#Create a test set. Note: this did not work because all of this are empty images. 
list_of_test_images = []
for image in list_of_images:
    if image not in list_of_filenames_in_annotation:
        list_of_test_images.append(image)
print(len(list_of_test_images), list_of_test_images)

66 ['P00000000879565588619499_250128_121532_974639.png', 'P00000000879565588619405_250205_121128_028609.png', 'P00000000879565588619499_250128_120809_166280.png', 'P00000000879565588619499_250128_120938_670498.png', 'P00000000879565588619405_250205_122724_571579.png', 'P00000000879565588619405_250205_121039_803010.png', 'P00000000879565588619405_250128_135954_218022.png', 'P00000000879565588619405_250205_114436_049870.png', 'P00000000879565588619499_250128_120711_552151.png', 'P00000000879565588619405_250205_122208_167810.png', 'P00000000879565588619405_250205_122604_143909.png', 'P00000000879565588619405_250205_122425_304502.png', 'P00000000879565588619405_250205_123453_863094.png', 'P00000000879565588619405_250205_114157_244332.png', 'P00000000879565588619405_250205_123321_266039.png', 'P00000000879565588619405_250205_114024_396231.png', 'P00000000879565588619405_250205_113834_955583.png', 'P00000000879565588619405_250205_110836_092183.png', 'P00000000879565588619405_250205_111308_95